# Diffusion Microscope — Experiment Runner

Runs all three experiments against GPT-2 and Pythia-410m and saves results to Google Drive.

**Runtime:** Select `Runtime → Change runtime type → T4 GPU` before running.

---
**Experiments:**
- **Exp 1 — Alpha compression visibility:** does alpha=1 produce more distinct images than alpha=1000? (both models, last layer)
- **Exp 2 — L0→L1 bottleneck:** what does the first transformer layer discard? (Pythia only, alpha=1, layers 0–3)
- **Exp 3 — Alpha sensitivity as novelty detector:** does LPIPS(alpha=1, alpha=1000) rank unusual prompts above common ones?

**Session persistence:** HuggingFace model cache and experiment results are stored on Drive — re-running skips completed work.

## 1 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — switch runtime to T4 GPU before continuing.')

In [ ]:
import os

# ── Edit these paths if needed ──────────────────────────────────────────────
REPO_URL    = 'https://github.com/leonorae/slicer'   # replace with your fork if needed
REPO_BRANCH = 'claude/geometric-visualization-pipeline-kHzEj'
REPO_DIR    = '/content/slicer'
DRIVE_BASE  = '/content/drive/MyDrive/diffusion_microscope'
# ────────────────────────────────────────────────────────────────────────────

HF_CACHE_DIR    = os.path.join(DRIVE_BASE, 'hf_cache')
RESULTS_GPT2    = os.path.join(DRIVE_BASE, 'experiment_results_nb_gpt2')
RESULTS_PYTHIA  = os.path.join(DRIVE_BASE, 'experiment_results_nb_pythia')

os.makedirs(HF_CACHE_DIR,   exist_ok=True)
os.makedirs(RESULTS_GPT2,   exist_ok=True)
os.makedirs(RESULTS_PYTHIA, exist_ok=True)

os.environ['HF_HOME']               = HF_CACHE_DIR
os.environ['TRANSFORMERS_CACHE']    = HF_CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR

print('Drive base :', DRIVE_BASE)
print('HF cache   :', HF_CACHE_DIR)
print('GPT-2 out  :', RESULTS_GPT2)
print('Pythia out :', RESULTS_PYTHIA)

## 2 · Clone repo and install

In [ ]:
import os

if os.path.isdir(REPO_DIR):
    print('Repo already cloned — pulling latest.')
    !git -C {REPO_DIR} fetch origin {REPO_BRANCH}
    !git -C {REPO_DIR} checkout {REPO_BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}

In [ ]:
# Colab already ships torch+CUDA — install everything else and avoid overwriting torch.
!pip install -q \
    open-clip-torch \
    diffusers \
    Pillow \
    lpips \
    datasets \
    nltk \
    sentencepiece \
    accelerate \
    scikit-learn \
    umap-learn

# Install the package itself (no deps — already installed above)
!pip install -q -e {REPO_DIR} --no-deps

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('Install complete.')

## 3 · Patch configs with Drive output paths

The repo ships configs pointing to local `./experiment_results_nb_*` dirs.
This cell rewrites the output paths to point at Drive so results persist.

In [ ]:
import json, shutil

def patch_config(src_name, out_dir):
    src = os.path.join(REPO_DIR, src_name)
    with open(src) as f:
        cfg = json.load(f)
    cfg['output']['base_dir'] = out_dir
    dst = os.path.join(REPO_DIR, src_name)
    with open(dst, 'w') as f:
        json.dump(cfg, f, indent=2)
    print(f'{src_name} → {out_dir}')

patch_config('experiment_config_nb_gpt2.json',   RESULTS_GPT2)
patch_config('experiment_config_nb_pythia.json',  RESULTS_PYTHIA)

## 4 · GPT-2 experiments

Covers **Exp 1** (alpha compression) and **Exp 3** (novelty detector).

Phases: `train` → `generate` → `grids` → `dashboard`

All phases are idempotent — if interrupted, re-run the cell and it will pick up from where it left off.

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_gpt2.json \
    --phase train

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_gpt2.json \
    --phase generate

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_gpt2.json \
    --phase grids

In [ ]:
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_gpt2.json \
    --phase dashboard
print(f'\nDashboard saved to Drive — download to view:\n  {RESULTS_GPT2}/dashboard.html')

## 5 · Pythia-410m experiments

Covers **Exp 1** (alpha compression, L23), **Exp 2** (L0→L1 bottleneck, L0-3), and **Exp 3** (novelty detector, L23).

Pythia-410m is ~800 MB. First run downloads it to the Drive HF cache.

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_pythia.json \
    --phase train

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_pythia.json \
    --phase generate

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_pythia.json \
    --phase grids

In [ ]:
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_pythia.json \
    --phase dashboard
print(f'\nDashboard saved to Drive — download to view:\n  {RESULTS_PYTHIA}/dashboard.html')

## 6 · Results

Run from here to review existing results without re-running the experiment.

In [ ]:
import pathlib, re, glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# ── Discover probe slugs from actual files on disk ───────────────────────────
def discover_slugs(results_dir):
    """Return sorted list of probe slugs that actually have images."""
    p = pathlib.Path(results_dir) / 'grids' / 'by_projection'
    if not p.exists():
        return []
    slugs = set()
    for proj_dir in p.iterdir():
        if proj_dir.is_dir():
            for slug_dir in proj_dir.iterdir():
                if slug_dir.is_dir():
                    slugs.add(slug_dir.name)
    return sorted(slugs)

gpt2_slugs   = discover_slugs(RESULTS_GPT2)
pythia_slugs = discover_slugs(RESULTS_PYTHIA)

print('GPT-2 slugs   :', gpt2_slugs or '(none found — run sections 4 first)')
print('Pythia slugs  :', pythia_slugs or '(none found — run sections 5 first)')

# ── Classify each slug into a tier ───────────────────────────────────────────
def classify_slug(slug):
    if any(w in slug for w in ['remembering', 'tuesday', 'midnight']):
        return 'unusual'
    if any(w in slug for w in ['cat', 'dog', 'house', 'tree']):
        return 'concrete'
    if any(w in slug for w in ['democracy', 'justice', 'entropy', 'grief', 'beauty', 'irony']):
        return 'abstract'
    return None

# ── Image lookup helpers ──────────────────────────────────────────────────────
def img_path(results_dir, alpha, slug, layer, cfg=7.5, seed=42):
    # Images are saved as L{layer:04d}_CFG{cfg:g}_seed{seed}.png
    p = (pathlib.Path(results_dir) / 'grids' / 'by_projection'
         / f'per_layer_alpha{alpha}' / slug / 'per_layer'
         / f'L{layer:04d}_CFG{cfg:g}_seed{seed}.png')
    return p if p.exists() else None

def grid_path(results_dir, alpha, slug, seed=42):
    p = (pathlib.Path(results_dir) / 'grids' / 'by_projection'
         / f'per_layer_alpha{alpha}' / slug
         / f'grid_seed{seed}.png')
    return p if p.exists() else None

print('\nSlug tiers:')
for s in gpt2_slugs:
    print(f'  {classify_slug(s) or "unclassified":10s}  {s}')


### Exp 1 — Alpha compression: does alpha=1 produce more distinct images?

**What's being tested:** High alpha (1000) regularises the Ridge projection heavily,
pulling all probe vectors toward the corpus mean — like lossy compression in embedding space.
Low alpha (1) preserves relative distances but amplifies noise in low-variance directions.

**What to look for:** Each figure shows the *same prompt* at alpha=1 (top row) vs alpha=1000 (bottom row).
- **Concrete prompts** (cat, dog, house, tree) should look *similar* across alphas — their
  signal is near the corpus mean, so compression doesn't destroy much.
- **Abstract prompts** (democracy, grief, entropy) should look *more different* — their
  distinguishing information lives in low-variance directions that high alpha kills.

The nn_recall@5 metric predicted this: last-layer nn_recall at alpha=1000 is lower for
abstract prompts (~0.35) than concrete (~0.5), meaning the compressor loses more
neighbourhood structure for abstract concepts.

In [ ]:
def show_alpha_comparison(results_dir, layer, slugs, title='', seeds=(42, 123, 777)):
    """2-row figure: alpha=1 top, alpha=1000 bottom, one column per slug.
    Uses the first seed that has images for both alphas."""
    # Find which slugs have images for both alphas
    available = []
    chosen_paths = {}  # slug → (path_alpha1, path_alpha1000)
    for slug in slugs:
        for seed in seeds:
            p1    = img_path(results_dir, 1,    slug, layer, seed=seed)
            p1000 = img_path(results_dir, 1000, slug, layer, seed=seed)
            if p1 and p1000:
                available.append(slug)
                chosen_paths[slug] = (p1, p1000)
                break

    if not available:
        print(f'No images found in {results_dir} for layer {layer} — check paths.')
        return

    n = len(available)
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 7), squeeze=False)
    fig.suptitle(title, fontsize=11, y=1.01)

    for col, slug in enumerate(available):
        p1, p1000 = chosen_paths[slug]
        for row, (p, row_label) in enumerate([(p1, 'α = 1'), (p1000, 'α = 1000')]):
            ax = axes[row][col]
            ax.imshow(Image.open(p))
            ax.axis('off')
            if row == 0:
                ax.set_title(slug.replace('_', ' '), fontsize=8)
            if col == 0:
                ax.set_ylabel(row_label, fontsize=9)

    plt.tight_layout()
    plt.show()

# Split GPT-2 slugs into concrete vs abstract
gpt2_concrete = [s for s in gpt2_slugs if classify_slug(s) == 'concrete']
gpt2_abstract = [s for s in gpt2_slugs if classify_slug(s) == 'abstract']

print('=== GPT-2 — Layer 11 (last layer) ===')
show_alpha_comparison(RESULTS_GPT2, layer=11, slugs=gpt2_concrete,
                      title='GPT-2 L11 — concrete prompts — alpha=1 vs 1000')
show_alpha_comparison(RESULTS_GPT2, layer=11, slugs=gpt2_abstract,
                      title='GPT-2 L11 — abstract prompts — alpha=1 vs 1000')

In [ ]:
pythia_concrete = [s for s in pythia_slugs if classify_slug(s) == 'concrete']
pythia_abstract = [s for s in pythia_slugs if classify_slug(s) == 'abstract']

print('=== Pythia-410m — Layer 23 (last layer) ===')
show_alpha_comparison(RESULTS_PYTHIA, layer=23, slugs=pythia_concrete,
                      title='Pythia L23 — concrete prompts — alpha=1 vs 1000')
show_alpha_comparison(RESULTS_PYTHIA, layer=23, slugs=pythia_abstract,
                      title='Pythia L23 — abstract prompts — alpha=1 vs 1000')

### Exp 2 — L0→L1 bottleneck in Pythia

**What's being tested:** At alpha=1000, Pythia L0 has the *highest* erank (494) and L1 dips
to 478 before subsequent layers rise again. This is the opposite of GPT-2 (where L0 is
singular/rank-deficient). The hypothesis: Pythia's embedding layer is maximally
diffuse, and the first transformer layer *compresses* it before later layers re-expand.

**What to look for:** Does L0 look more diffuse or generic than L1?
If the bottleneck hypothesis is right, L0 images should look less structured than L1 —
more like random textures or colour fields — because L0 has no transformer computation
behind it, just raw token embeddings.

**Important caveat:** the erank dip only appears at alpha=1000. At alpha=1 (shown
below), L0 has the *lowest* erank (354) and rises monotonically. The experiment tests
both alphas to distinguish architectural signal from regularisation artefact.

In [ ]:
def show_layer_sweep(results_dir, alpha, layers, slug, title='', seeds=(42, 123, 777)):
    """One image per layer for a single probe."""
    valid = []
    for L in layers:
        for seed in seeds:
            p = img_path(results_dir, alpha, slug, L, seed=seed)
            if p:
                valid.append((L, p))
                break

    if not valid:
        print(f'No images for slug "{slug}" at alpha={alpha} in {results_dir}')
        return

    fig, axes = plt.subplots(1, len(valid), figsize=(3 * len(valid), 4), squeeze=False)
    fig.suptitle(title or f'"{slug.replace("_", " ")}" — L{layers[0]}→L{layers[-1]} at alpha={alpha}',
                 fontsize=10)
    for i, (L, p) in enumerate(valid):
        axes[0][i].imshow(Image.open(p))
        axes[0][i].axis('off')
        axes[0][i].set_title(f'L{L}', fontsize=9)
    plt.tight_layout()
    plt.show()

# Pick one concrete and one abstract probe for the sweep
ref_concrete = next((s for s in pythia_slugs if classify_slug(s) == 'concrete'), None)
ref_abstract = next((s for s in pythia_slugs if classify_slug(s) == 'abstract'), None)

for slug in [ref_concrete, ref_abstract]:
    if not slug:
        continue
    print(f'\n--- "{slug.replace("_", " ")}" ---')
    show_layer_sweep(RESULTS_PYTHIA, alpha=1,    layers=[0, 1, 2, 3],
                     title=f'"{slug.replace("_", " ")}" at alpha=1 — raw embedding → first 3 transformer layers')
    show_layer_sweep(RESULTS_PYTHIA, alpha=1000, layers=[0, 1, 2, 3],
                     title=f'"{slug.replace("_", " ")}" at alpha=1000 — same layers with heavy compression')

### Exp 3 — Alpha sensitivity as novelty detector

**What's being tested:** Do unusual prompts change *more* between alpha=1 and alpha=1000
than common ones? The hypothesis: unusual prompts have their distinguishing information
concentrated in low-variance directions that high alpha kills. Common prompts live near
the corpus mean — their projection is insensitive to the compression.

**Tiers:**
- *Common-concrete* (should be alpha-insensitive): cat, dog, house, tree
- *Common-abstract* (moderate sensitivity): democracy, justice, beauty, grief
- *Unusual* (should be alpha-sensitive): "the feeling of almost remembering",
  "the color of Tuesday", "entropy at midnight"

**What to look for:** LPIPS(alpha=1 image, alpha=1000 image) — higher = more different.
If the hypothesis holds, the bar chart should show: unusual > abstract > concrete.
If the bars are flat across tiers, compression is not tier-sensitive and the
alpha-as-novelty-detector idea doesn't hold for this corpus and model.

**Confound to watch:** low-variance directions in the projection ≠ semantically unusual
prompts. They could just be directions underrepresented in the training corpus.

In [ ]:
import torch
import lpips as lpips_lib

_lpips_fn = None

def get_lpips():
    global _lpips_fn
    if _lpips_fn is None:
        _lpips_fn = lpips_lib.LPIPS(net='alex', verbose=False)
    return _lpips_fn

def compute_lpips(path_a, path_b):
    """LPIPS perceptual distance between two image files."""
    def load_tensor(p):
        img = Image.open(p).convert('RGB').resize((256, 256))
        t = torch.tensor(np.array(img), dtype=torch.float32)
        t = t.permute(2, 0, 1) / 127.5 - 1.0  # [-1, 1]
        return t.unsqueeze(0)
    fn = get_lpips()
    with torch.no_grad():
        return fn(load_tensor(path_a), load_tensor(path_b)).item()

def lpips_alpha_pair(results_dir, layer, slug, seeds=(42, 123, 777)):
    """Mean LPIPS between alpha=1 and alpha=1000 across available seeds."""
    vals = []
    for seed in seeds:
        p1    = img_path(results_dir, 1,    slug, layer, seed=seed)
        p1000 = img_path(results_dir, 1000, slug, layer, seed=seed)
        if p1 and p1000:
            vals.append(compute_lpips(str(p1), str(p1000)))
    return np.mean(vals) if vals else None

# ── Compute LPIPS for all slugs, both models ─────────────────────────────────
tier_order  = ['concrete', 'abstract', 'unusual']
tier_colors = ['#4CAF50', '#FF9800', '#F44336']

results_lpips = {}  # model_label → {slug: lpips_value}

for model_label, results_dir, last_layer, slugs in [
    ('GPT-2 L11',       RESULTS_GPT2,   11, gpt2_slugs),
    ('Pythia-410m L23', RESULTS_PYTHIA, 23, pythia_slugs),
]:
    print(f'Computing LPIPS for {model_label}...')
    scores = {}
    for slug in slugs:
        v = lpips_alpha_pair(results_dir, last_layer, slug)
        if v is not None:
            scores[slug] = v
            tier = classify_slug(slug) or 'other'
            print(f'  {tier:10s}  {slug:42s}  LPIPS = {v:.4f}')
    results_lpips[model_label] = scores

print('Done.')

In [ ]:
fig, axes = plt.subplots(1, len(results_lpips), figsize=(7 * len(results_lpips), 5),
                         squeeze=False)

for ax, (model_label, scores) in zip(axes[0], results_lpips.items()):
    tier_vals = {t: [] for t in tier_order}
    for slug, v in scores.items():
        t = classify_slug(slug)
        if t in tier_vals:
            tier_vals[t].append(v)

    means = [np.mean(tier_vals[t]) if tier_vals[t] else 0 for t in tier_order]
    stds  = [np.std(tier_vals[t])  if len(tier_vals[t]) > 1 else 0 for t in tier_order]

    bars = ax.bar(tier_order, means, yerr=stds, color=tier_colors,
                  alpha=0.85, capsize=5, edgecolor='white', linewidth=0.5)

    # Annotate individual points
    for t_idx, tier in enumerate(tier_order):
        for v in tier_vals[tier]:
            ax.plot(t_idx, v, 'o', color='white', markersize=4, alpha=0.7)

    ax.set_title(f'{model_label}\nLPIPS(α=1, α=1000) by probe tier', fontsize=10)
    ax.set_ylabel('LPIPS  (higher = images differ more between alphas)')
    ax.set_ylim(0, max(max(means) * 1.4, 0.05))
    ax.grid(True, alpha=0.25, axis='y')
    ax.axhline(np.mean([v for v in scores.values()]), color='white',
               linestyle='--', linewidth=0.8, alpha=0.5, label='mean')

plt.suptitle('Exp 3: Alpha sensitivity by tier\n'
             'If unusual > abstract > concrete, alpha compresses novelty-bearing directions most',
             fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side images for highest and lowest LPIPS probes
for model_label, results_dir, last_layer, slugs in [
    ('GPT-2 L11',       RESULTS_GPT2,   11, gpt2_slugs),
    ('Pythia-410m L23', RESULTS_PYTHIA, 23, pythia_slugs),
]:
    scores = results_lpips.get(model_label, {})
    if not scores:
        continue
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print(f'\n=== {model_label} — highest and lowest LPIPS pairs ===')
    for slug, v in ranked[:2] + ranked[-2:]:
        show_alpha_comparison(results_dir, last_layer, [slug],
                              title=f'"{slug.replace("_", " ")}" — LPIPS={v:.3f} ({classify_slug(slug)})')

### Full grid PNGs — best overview

Each grid image shows **all layers × all seeds** for a single probe at one alpha setting.
This is the most information-dense view — good for spotting whether layers produce
coherent variation or whether the images look random.

In [ ]:
from IPython.display import display as ipy_display

def show_grid_images(results_dir, alpha, model_label):
    pattern = str(pathlib.Path(results_dir) / 'grids' / 'by_projection'
                  / f'per_layer_alpha{alpha}' / '*' / 'grid_seed*.png')
    grid_paths = sorted(glob.glob(pattern))
    if not grid_paths:
        print(f'No grid PNGs found for {model_label} alpha={alpha}')
        return
    for p in grid_paths:
        slug = pathlib.Path(p).parent.name
        tier = classify_slug(slug) or 'other'
        print(f'  [{tier}]  {slug.replace("_", " ")}  (alpha={alpha})')
        ipy_display(Image.open(p))

for alpha in [1, 1000]:
    print(f'\n══ GPT-2   alpha={alpha} ══════════════════════════════════════')
    show_grid_images(RESULTS_GPT2, alpha, 'GPT-2')

for alpha in [1, 1000]:
    print(f'\n══ Pythia  alpha={alpha} ══════════════════════════════════════')
    show_grid_images(RESULTS_PYTHIA, alpha, 'Pythia-410m')

## 7 · Save summary

Results are already on Drive (output dirs point there). This cell writes a short summary of what ran.

In [ ]:
import datetime

summary_path = os.path.join(DRIVE_BASE, 'run_summary.txt')
lines = [
    f'Run completed: {datetime.datetime.now().isoformat()}',
    f'Branch: {REPO_BRANCH}',
    '',
    'GPT-2 results:',
]
for root, dirs, files in os.walk(RESULTS_GPT2):
    png_count = sum(1 for f in files if f.endswith('.png'))
    if png_count:
        lines.append(f'  {os.path.relpath(root, RESULTS_GPT2)}: {png_count} images')

lines.append('')
lines.append('Pythia-410m results:')
for root, dirs, files in os.walk(RESULTS_PYTHIA):
    png_count = sum(1 for f in files if f.endswith('.png'))
    if png_count:
        lines.append(f'  {os.path.relpath(root, RESULTS_PYTHIA)}: {png_count} images')

if results_lpips:
    lines.append('')
    lines.append('Exp 3 LPIPS summary (alpha=1 vs alpha=1000):')
    for model_label, scores in results_lpips.items():
        lines.append(f'  {model_label}:')
        for slug, v in sorted(scores.items(), key=lambda x: -x[1]):
            tier = classify_slug(slug) or 'other'
            lines.append(f'    {tier:10s} {slug:42s} {v:.4f}')

summary = '\n'.join(lines)
print(summary)
with open(summary_path, 'w') as f:
    f.write(summary)
print(f'\nSummary written to {summary_path}')